# 02 - Pipeline 与 ColumnTransformer

## 学习目标
- 掌握 Pipeline 的使用
- 学会用 ColumnTransformer 处理混合类型数据
- 构建完整的预处理流水线

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score

## 1. 基本 Pipeline

In [ ]:
# 创建 Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 使用 Pipeline
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print(f'R² Score: {score:.4f}')

## 2. ColumnTransformer

In [ ]:
# 创建混合类型数据
df = pd.DataFrame({
    'age': [25, 30, 35, np.nan, 45, 50, 28, 33],
    'income': [50000, 60000, np.nan, 80000, 90000, 70000, 55000, 65000],
    'city': ['Beijing', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai'],
    'education': ['Bachelor', 'Master', 'Bachelor', 'PhD', 'Master', 'Bachelor', 'PhD', 'Master'],
    'target': [100, 150, 120, 200, 180, 130, 110, 160]
})

X = df.drop('target', axis=1)
y = df['target']

# 定义数值型和类别型特征
numeric_features = ['age', 'income']
categorical_features = ['city', 'education']

# 数值型预处理
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 类别型预处理
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('ColumnTransformer 创建完成')

## 3. 完整 Pipeline

In [ ]:
# 完整 Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 交叉验证
scores = cross_val_score(full_pipeline, X, y, cv=3, scoring='r2')
print(f'CV R²: {scores.mean():.4f} ± {scores.std():.4f}')

## 小结
- Pipeline 防止数据泄漏
- ColumnTransformer 处理混合类型
- 完整的流水线便于部署